# Améliorations — reconstruire la trajectoire de chaque patient

Après le guide, notre meilleur modèle était à **7,43** (RMSE, validation par patient). Ce notebook explique comment nous sommes descendus à **3,37** (Kaggle : 3,339 public, 3,192 privé).

Les variables et le modèle sont dans `features.py` (racine du repo). Ce notebook remplace les scripts `archives_py/experiment_day2_step8*.py` à `step10.py`.

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
if not os.path.exists("data"):   # le notebook est dans notebooks_finaux/ : on se place à la racine du repo
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
from sklearn.base import clone
import skore
from skore import evaluate

skore.configuration.show_progress = False  # pas de barres de progression dans le notebook
PUSH = False  # les rapports sont déjà sur Skore Hub : True seulement pour les renvoyer

def push(key, report):
    """Envoie un rapport sur Skore Hub (workspace lu dans .skore) si PUSH vaut True."""
    if not PUSH:
        print(f"(PUSH=False) rapport {key!r} déjà sur le Hub, non renvoyé")
        return
    from skore import Project, login
    cfg = json.load(open(".skore", encoding="utf-8"))
    os.environ["SKORE_HUB_URI"], os.environ["SKORE_HUB_API_KEY"] = cfg["hub_url"], cfg["api_key"]
    login(mode="hub")
    Project(name="bobathon-esilv", mode="hub", workspace=cfg["workspace"]).put(key, report)

In [ ]:
def cv_rmse(report, data_source="test"):
    # RMSE de chacun des 5 folds d'un rapport de validation croisée
    return report.metrics.rmse(aggregate=None, data_source=data_source).iloc[0].to_numpy(dtype=float)

def show(name, report):
    # RMSE moyen ± écart entre folds, et écart train/test (surapprentissage)
    test, train = cv_rmse(report), cv_rmse(report, "train")
    print(f"{name:<22} test {test.mean():.3f} ± {test.std(ddof=1):.3f}   train {train.mean():.3f}   écart {test.mean() - train.mean():+.3f}")
    return test

In [ ]:
from cv_splits import load_visits, make_grouped_cv_splits

visits = load_visits()
y = visits["target"]
cv_splits = make_grouped_cv_splits(visits)   # mêmes 5 folds par patient que le Day 2

## 1. L'idée clé

Pour un patient, le vrai score OFF évolue **lentement et régulièrement** avec l'âge. Vérification : on ajuste une courbe du 2ᵉ degré de `target` en fonction de l'âge, patient par patient, et on mesure l'erreur restante.

In [ ]:
def within_patient_rmse(deg):
    # erreur restante quand on ajuste une courbe de degré deg sur chaque patient
    res = []
    for _, g in visits.groupby("patient_id"):
        c = np.polyfit(g["age"], g["target"], deg)
        res.append(g["target"] - np.polyval(c, g["age"]))
    return np.sqrt(np.mean(np.concatenate(res) ** 2))

for deg in [1, 2]:
    print(f"courbe de degré {deg} par patient : erreur {within_patient_rmse(deg):.2f}")

Une erreur de **0,33** : la cible est presque exactement une courbe lisse par patient. Les mesures `on` et `off`, elles, sont bruitées et biaisées :
- `off` est presque toujours **en dessous** du vrai score (le médicament restant fait baisser le score) ;
- `on` vaut environ 44 à 72 % du vrai score selon le délai depuis la prise.

Chaque patient du **test** a aussi ~8 visites : on peut combiner ses visites pour reconstruire sa trajectoire. **Règle anti-fuite :** on n'utilise jamais `target` dans une variable (il n'existe pas dans le test), et jamais `patient_id` comme variable.

In [ ]:
# écart off mesuré - vrai score : surtout négatif, off sous-estime
(visits["off"] - visits["target"]).describe().round(2)

## 2. Les variables par patient (`features.py`)

`make_features` s'applique **de la même façon** au train et au test. Familles de variables :

| Famille | Exemples | Idée |
|---|---|---|
| Position dans le temps | `dur`, `t_first`, `visit_rank`, `n_visits` | où en est le patient dans sa maladie |
| Manques | `off_missing`, `on_missing` | un examen sauté est une information |
| `on` corrigé | `on_corr` | `on` divisé par l'effet attendu du médicament selon le délai |
| Statistiques du patient | `off_pmean`, `off_pmax`, `off_lin`, `off_quad`… | moyenne, max, droite et courbe ajustées sur ses visites |
| Visites voisines | `off_prev`, `off_next`… | mesure précédente et suivante |
| Trajectoire | `traj`, `traj_slope` | droite ajustée sur `off` + 5,9 et `on_corr`, pente tirée vers 2,8 points/an |
| Enveloppe supérieure | `off_env`, `comb_env`… | droite qui suit le **haut** des mesures `off` (puisque `off` sous-estime) |

In [ ]:
from features import make_features, TwoStageRegressor

X_feat = make_features(visits.drop(columns="target"))   # target retiré avant : il ne peut pas servir
# variables d'enveloppe, ajoutées seulement au modèle 10
ENV = ["off_env", "on_env", "comb_env", "off_minus_env", "traj_minus_env"]
print(X_feat.shape[1], "variables :", ", ".join(X_feat.columns))

## 3. Modèle 08 — HGBR sur les variables par patient

Même HGBR qu'au Day 2, un peu plus régularisé (`min_samples_leaf=40`, `l2_regularization=1.0`), sans les variables d'enveloppe (ajoutées plus tard).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgbr = HistGradientBoostingRegressor(random_state=0, max_iter=800, learning_rate=0.03,
                                     min_samples_leaf=40, l2_regularization=1.0)
report_08 = evaluate(hgbr, X_feat.drop(columns=ENV), y, splitter=cv_splits)   # sans enveloppe
show("08 variables patient", report_08)
push("08_patient_feats_gcv", report_08)

## 4. Modèle 09 — empilement en deux étages + lissage

`TwoStageRegressor` (dans `features.py`) :
1. **Étage 1** : le HGBR du modèle 08 prédit chaque visite. Ses prédictions sur le train sont faites **hors échantillon** (validation croisée interne par patient), sinon l'étage 2 apprendrait sur des prédictions trop parfaites.
2. **Étage 2** : un second HGBR reçoit en plus la trajectoire de ces prédictions pour le patient (moyenne, droite, courbe).
3. **Lissage** : les prédictions finales d'un patient sont remplacées par une courbe du 2ᵉ degré, puis ramenées entre 0 et 132.

`patient_id` est passé en colonne d'aide (pour regrouper les visites) mais retiré avant chaque modèle. ⏱ Quelques minutes.

In [ ]:
X_in = X_feat.copy()
X_in["patient_id"] = visits["patient_id"].to_numpy()   # colonne d'aide, retirée dans TwoStageRegressor

report_09 = evaluate(TwoStageRegressor(), X_in.drop(columns=ENV), y, splitter=cv_splits)
show("09 empilement", report_09)
push("09_stack_gcv", report_09)

## 5. Modèle 10 — ajout de l'enveloppe supérieure

Puisque `off` est presque toujours **sous** le vrai score, la droite qui suit le haut des mesures `off` d'un patient est une meilleure estimation que leur moyenne. ⏱ Quelques minutes.

In [ ]:
report_10 = evaluate(TwoStageRegressor(), X_in, y, splitter=cv_splits)
folds_10 = show("10 + enveloppe", report_10)
print("10 bat 09 sur chaque fold :", bool((folds_10 < cv_rmse(report_09)).all()))
push("10_env_stack_gcv", report_10)

## 6. Soumission du modèle 10
Réentraînement sur tout le train, mêmes variables sur le test, fichier dans l'ordre de `sample_submission.csv`. ⏱ Une minute.

In [ ]:
X_test = pd.read_csv("data/X_test.csv")
sample = pd.read_csv("data/sample_submission.csv")

X_test_in = make_features(X_test)   # même fonction que pour le train
X_test_in["patient_id"] = X_test["patient_id"].to_numpy()
X_test_in = X_test_in[X_in.columns]   # mêmes colonnes, même ordre que le train

final = clone(TwoStageRegressor()).fit(X_in, y)
submission = pd.DataFrame({"Index": X_test["Index"], "target": final.predict(X_test_in)})
submission = submission.set_index("Index").reindex(sample["Index"]).reset_index()   # ordre de sample_submission
# contrôles : 11 013 lignes, pas de NaN, valeurs entre 0 et 132
assert len(submission) == 11013 and submission["target"].notna().all() and submission["target"].between(0, 132).all()
submission.to_csv("submissions/10_env_stack.csv", index=False)
submission["target"].describe().round(2)

## 7. Bilan

In [ ]:
bilan = pd.DataFrame({
    "modèle": ["01 dummy", "02 ridge", "07 tabular_pipeline (fin du guide)", "08 variables patient", "09 empilement + lissage", "10 + enveloppe"],
    "RMSE GroupKFold": [16.50, 10.44, 7.43, cv_rmse(report_08).mean(), cv_rmse(report_09).mean(), cv_rmse(report_10).mean()],
})
bilan["RMSE GroupKFold"] = bilan["RMSE GroupKFold"].round(3)
bilan

| | RMSE |
|---|---|
| Validation par patient (GroupKFold) | 3,369 |
| Découpage aléatoire | 3,074 (trop optimiste) |
| **Kaggle public** | **3,339** |
| **Kaggle privé** | **3,192** |

---
## Ce qu'il faut retenir

1. **La bonne validation.** La GroupKFold par patient prédit le score Kaggle à 0,03 près ; le découpage aléatoire est trop optimiste. Tous nos choix ont été faits sur la GroupKFold, jamais sur le classement public.
2. **Le gros gain vient des variables, pas du modèle.** Passer à des variables par patient divise l'erreur par deux (7,43 → 3,74) : on traite enfin la progression dans le temps, problème cité dans `docs/CONTEXT.md`.
3. **Comprendre comment les mesures sont biaisées paie.** `off` sous-estime le vrai score, d'où l'enveloppe supérieure ; `on` dépend du délai depuis la prise, d'où `on_corr`.
4. **Pas de fuite.** Aucune variable n'utilise `target` ; `patient_id` et `Index` ne sont jamais des variables (l'étape 6 a montré qu'ils font surapprendre) ; les prédictions de l'étage 1 sont hors échantillon.
5. **Surapprentissage sous contrôle.** Le score varie très peu d'un fold à l'autre (± 0,05) et le score privé (3,19) est même meilleur que le public.